In [ ]:
import simpy
import pandas as pd
import numpy as np
import random

# --- 시스템 설정 ---
RACK_START_Y, RACK_END_Y = 4.0, 15.1
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2]
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]}
CTU_SPEED_XY, CTU_SPEED_Z = 1.8, 0.5
TOTE_CAPACITY = 7

class CTU:
    def __init__(self, env, id, warehouse, dispatch_rule):
        self.env, self.id, self.warehouse = env, id, warehouse
        self.dispatch_rule = dispatch_rule
        self.pos = STATIONS['S01']
        self.total_distance, self.completed_jobs = 0, 0
        self.busy = False
        self.process = env.process(self.run())

    def calculate_travel_time(self, target_pos):
        dx, dy, dz = abs(target_pos[0]-self.pos[0]), abs(target_pos[1]-self.pos[1]), abs(target_pos[2]-self.pos[2])
        return max((dx + dy) / CTU_SPEED_XY, dz / CTU_SPEED_Z), dx + dy + dz

    def run(self):
        while True:
            if self.warehouse.job_queue:
                self.busy = True
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    for job in batch:
                        t, d = self.calculate_travel_time(job['pos'])
                        yield self.env.timeout(t)
                        self.pos, self.total_distance, self.completed_jobs = job['pos'], self.total_distance + d, self.completed_jobs + 1
                    
                    d1 = np.linalg.norm(np.array(self.pos)-np.array(STATIONS['S01']))
                    d2 = np.linalg.norm(np.array(self.pos)-np.array(STATIONS['S02']))
                    near = 'S01' if d1 < d2 else 'S02'
                    t, d = self.calculate_travel_time(STATIONS[near])
                    yield self.env.timeout(t)
                    self.pos, self.total_distance = STATIONS[near], self.total_distance + d
                self.busy = False
            else:
                yield self.env.timeout(1)

class Warehouse:
    def __init__(self, env, bom_df, plan_df, slotting_rule):
        self.env, self.bom_df, self.plan_df = env, bom_df, plan_df
        self.job_queue = []
        self.inventory = self.apply_slotting_rule(slotting_rule)

    def apply_slotting_rule(self, rule_type):
        inventory = {}
        all_parts = sorted(self.bom_df['part_code'].unique())
        
        if rule_type == 'random':
            for part in all_parts:
                inventory[part] = [round(random.choice(AISLE_X), 2), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), round(random.uniform(0.5, 3.5), 2)]
        
        elif rule_type == 'abc':
            freq = self.plan_df['부품코드'].value_counts().to_dict()
            sorted_parts = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)
            for i, part in enumerate(sorted_parts):
                y_pos = RACK_START_Y + (i / len(all_parts)) * (RACK_END_Y - RACK_START_Y)
                inventory[part] = [random.choice(AISLE_X), round(y_pos, 2), round(random.uniform(0.5, 3.5), 2)]

        elif rule_type == 'characteristic':
            for i, part in enumerate(all_parts):
                aisle = AISLE_X[i % len(AISLE_X)]
                inventory[part] = [aisle, round(random.uniform(RACK_START_Y, RACK_END_Y), 2), round(random.uniform(0.5, 3.5), 2)]

        elif rule_type == 'kitting':
            order_groups = self.plan_df.groupby('No.')['부품코드'].apply(list).tolist()
            assigned, idx = set(), 0
            for group in order_groups:
                for part in group:
                    if part not in assigned:
                        aisle = AISLE_X[idx % len(AISLE_X)]
                        y_pos = RACK_START_Y + (idx / len(all_parts)) * (RACK_END_Y - RACK_START_Y)
                        inventory[part] = [aisle, round(y_pos, 2), round(random.uniform(0.5, 3.5), 2)]
                        assigned.add(part)
                        idx += 1
            for part in [p for p in all_parts if p not in assigned]:
                inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), round(random.uniform(0.5, 3.5), 2)]
        
        return inventory

    def apply_dispatching(self, ctu, rule):
        if not self.job_queue: return []
        
        # 1. Seed 선택
        if rule == 'nearest':
            self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))
        elif rule == 'density':
            def get_density(job):
                return sum(1 for other in self.job_queue if np.linalg.norm(np.array(job['pos']) - np.array(other['pos'])) < 3.0)
            self.job_queue.sort(key=get_density, reverse=True)
        elif rule == 'load_balance':
            self.job_queue.sort(key=lambda j: j['time'])

        # 2. 배치 구성
        batch = [self.job_queue.pop(0)]
        if self.job_queue:
            self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(batch[0]['pos']) - np.array(j['pos'])))
            extra = min(len(self.job_queue), TOTE_CAPACITY - 1)
            batch.extend(self.job_queue[:extra])
            self.job_queue = self.job_queue[extra:]
        return batch

def monitor(env, warehouse, gen_proc, ctus):
    yield gen_proc
    while warehouse.job_queue or any(c.busy for c in ctus):
        yield env.timeout(1)

def run_simulation(slotting_rule, dispatch_rule, bom_df, plan_df):
    env = simpy.Environment()
    warehouse = Warehouse(env, bom_df, plan_df, slotting_rule)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    def job_gen(env, wh):
        for _, group in wh.plan_df.groupby('No.'):
            yield env.timeout(random.uniform(5, 10))
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    gen_proc = env.process(job_gen(env, warehouse))
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    return {'Slotting': slotting_rule, 'Dispatching': dispatch_rule, 
            'Time': round(env.now, 1), 'Distance': round(sum(c.total_distance for c in ctus), 1)}

def main():
    try:
        bom_df = pd.read_excel('BOM.xlsx')
        plan_df = pd.read_csv('outbound_plan.csv')
        
        slotting_rules = ['random', 'abc', 'characteristic', 'kitting']
        dispatch_rules = ['nearest', 'density', 'load_balance']
        results = []

        print("전체 시나리오 시뮬레이션 시작...")
        for s_rule in slotting_rules:
            for d_rule in dispatch_rules:
                res = run_simulation(s_rule, d_rule, bom_df, plan_df)
                results.append(res)
                print(f" - [Slot: {s_rule.upper():<8} | Disp: {d_rule.upper():<12}] 완료")

        # 결과 요약 출력
        df_res = pd.DataFrame(results)
        print("\n" + "="*80)
        print(f"{'Slotting':<15} | {'Dispatching':<15} | {'Time (s)':<12} | {'Dist (m)':<12}")
        print("-" * 80)
        # 시간순으로 정렬하여 가장 빠른 케이스가 상단에 오도록 함
        df_res = df_res.sort_values(by='Time')
        for _, r in df_res.iterrows():
            print(f"{r['Slotting'].upper():<15} | {r['Dispatching'].upper():<15} | {r['Time']:<12.1f} | {r['Distance']:<12.1f}")
        print("="*80)
        print(f"BEST CASE: Slotting [{df_res.iloc[0]['Slotting']}], Dispatching [{df_res.iloc[0]['Dispatching']}]")

    except Exception as e:
        print(f"오류 발생: {e}")

if __name__ == "__main__":
    main()